Este notebook sirve para probar rápidamente los modelos antes de correr el script completo.

Código inicial:

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from src.data_loader import load_cleveland
from src.preprocessing import build_preprocessor
from src.evaluate import evaluate_classification

Cargar datos:

In [2]:
df = load_cleveland()
df = df.drop(columns=["source"])

X = df.drop(columns=["target"])
y = df["target"]

Dividir datos:

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

Modelos base:

In [4]:
models = {
    "Regresión Logística": LogisticRegression(max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Árbol de Decisión": DecisionTreeClassifier(random_state=42, max_depth=4),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=200),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

Entrenar y comparar:

In [5]:
results = []

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", build_preprocessor()),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline.named_steps["model"], "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
    else:
        y_proba = None

    results.append(
        evaluate_classification(
            model_name,
            y_test,
            y_pred,
            y_proba
        )
    )

pd.DataFrame(results).sort_values(by="recall", ascending=False)

,modelo,accuracy,precision,recall,f1,roc_auc
0,Regresión Logística,0.868852,0.812500,0.928571,0.866667,0.957792
1,KNN,0.901639,0.866667,0.928571,0.896552,0.952922
3,Random Forest,0.885246,0.838710,0.928571,0.881356,0.941017
4,Gradient Boosting,0.885246,0.838710,0.928571,0.881356,0.943723
2,Árbol de Decisión,0.786885,0.741935,0.821429,0.779661,0.859848
